# Notebook 1 — Data Acquisition
**Project:** DRC Flood Extent Mapping (Jan 2025 – Apr 2026)  
**AOI:** North Kivu, South Kivu, Ituri — Eastern DRC  
**Primary source:** Sentinel-1 SAR (cloud-independent)  
**Secondary source:** Sentinel-2 optical (cloud-permitting only)

## Strengths
- Sentinel-1 SAR penetrates persistent tropical cloud cover — critical for DRC
- STAC-based discovery returns only COG assets: no full-scene downloads needed
- Monthly cadence captures seasonal flood cycles

## Limitations
- Sentinel-1 revisit over eastern DRC is ~12 days; a flood peaking between passes will be underestimated or missed in monthly composites
- Sentinel-2 will have very low scene availability due to persistent cloud cover; treat as supplementary only
- Microsoft Planetary Computer requires a (free) API token for signed asset access

In [ ]:
import sys
sys.path.append('..')

from src.acquisition import load_config, monthly_date_ranges, search_sentinel1, search_sentinel2
import geopandas as gpd
import folium
from shapely.geometry import box

cfg = load_config('../config/config.yaml')
print('Config loaded.')
print(f"AOI: {cfg['aoi']['name']}")
print(f"Period: {cfg['temporal']['start']} to {cfg['temporal']['end']}")

In [ ]:
# Visualise AOI
b = cfg['aoi']['bbox']
aoi = box(b['west'], b['south'], b['east'], b['north'])
gdf = gpd.GeoDataFrame(geometry=[aoi], crs='EPSG:4326')

m = folium.Map(location=[(-5.9 + 3.0) / 2, (26.8 + 30.8) / 2], zoom_start=6)
folium.GeoJson(gdf).add_to(m)
folium.LayerControl().add_to(m)
m

In [ ]:
# Generate monthly date ranges for the full temporal window
date_ranges = monthly_date_ranges(cfg)
print(f'{len(date_ranges)} monthly windows:')
for s, e in date_ranges:
    print(f'  {s} → {e}')

In [ ]:
# Search Sentinel-1 for first month as a test
start, end = date_ranges[0]
s1_items = search_sentinel1(cfg, start, end)
print(f'\nFirst item assets: {list(s1_items[0].assets.keys()) if s1_items else "none"}')

In [ ]:
# Search Sentinel-2 — expect low availability due to cloud cover
s2_items = search_sentinel2(cfg, start, end)
print(f'Note: low S2 count expected — persistent cloud cover over eastern DRC')